# Progetto: "AI Ghostwriter" - Adattamento e Tuning

**Scenario:** la demo su Shakespeare ha funzionato. Il cliente (casa editrice "Dante & Co.") vuole generare testo in **italiano** nello stile della **Divina Commedia**, e poter calibrare la creatività dell'AI (oggi a volte "troppo caotica", a volte "troppo ripetitiva").

**Obiettivi**
1. **Data Swapping** — sostituire Shakespeare con Dante
2. **Hyperparameter Tuning** — trovare la temperatura giusta tra grammatica e creatività
3. **Prompt Engineering** — adattare `start_string` al nuovo contesto

## 0. Import

In [1]:
import os
import time

import numpy as np
import tensorflow as tf
import keras
import torch  # usato da generate_text (torch.no_grad) nel capitolo 1
import requests
import urllib3

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

TensorFlow Version: 2.21.0
Keras backend: tensorflow


---
# Capitolo 1 — Prototipo Shakespeare (codice del docente)

Esegui il codice base così com'è e genera una frase di test.

- **Analisi:** osserva l'output di `elabora_dizionari`. Quanti caratteri unici usava Shakespeare?
- *Annota il numero, servirà per il confronto.*

In [2]:
# --- CONFIGURAZIONE BACKEND E PERFORMANCE ---
# Nel codice originale os.environ["KERAS_BACKEND"] = "torch" viene impostato DOPO `import keras`:
# a quel punto il backend è già scelto, quindi la riga non ha effetto e si gira su TensorFlow.
# Qui la lasciamo commentata per lo stesso motivo (e la soluzione Dante usa tf.keras).
# os.environ["KERAS_BACKEND"] = "torch"

# mixed_float16 serve su GPU; siamo su CPU, e la policy è globale: finirebbe anche nel modello di Dante.
# keras.mixed_precision.set_global_policy("mixed_float16")

# --- ACQUISIZIONE E PRE-PROCESSAMENTO DEI DATI ---
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
vocab = sorted(set(text))
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [6]:
char_unici = len(set(text))
print('Caratteri unici in Shakespeare:', char_unici)

Caratteri unici in Shakespeare: 65


In [7]:
# --- CREAZIONE DEL DATASET OTTIMIZZATO ---
seq_length = 250
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

BATCH_SIZE = 128
BUFFER_SIZE = 10000
dataset = dataset.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

In [8]:
# --- DEFINIZIONE DEL MODELLO (CON 3 LAYER GRU) ---
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = keras.Sequential([
        # Embedding Layer
        keras.layers.Embedding(vocab_size, embedding_dim),

        # 1° Layer GRU
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False, recurrent_initializer='glorot_uniform'),

        # 2° Layer GRU (Intermedio - Modifica richiesta)
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),

        # 3° Layer GRU (Aggiunto per aumentare la capacità di apprendimento)
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),

        # Dense Layer (Logits)
        keras.layers.Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)

In [9]:
# --- TRAINING ---
model.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

# Originale: epochs=50 (pensato per GPU). Su CPU scelgo 10 solo per verificare il prototipo.
EPOCHS_SHAKESPEARE = 10
model.fit(dataset, epochs=EPOCHS_SHAKESPEARE)

Epoch 1/10


c:\Users\veron\miniconda3\envs\epicode-io\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


34/34 ━━━━━━━━━━━━━━━━━━━━ 317s 9s/step - loss: 3.4024
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 311s 9s/step - loss: 2.5062
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 309s 9s/step - loss: 2.1426
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 313s 9s/step - loss: 1.9032
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 307s 9s/step - loss: 1.7088
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 306s 9s/step - loss: 1.5696
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 300s 9s/step - loss: 1.4723
Epoch 8/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 300s 9s/step - loss: 1.3996
Epoch 9/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 299s 9s/step - loss: 1.3468
Epoch 10/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 299s 9s/step - loss: 1.3008


In [10]:
# --- FUNZIONE DI GENERAZIONE E ANALISI DELLA TEMPERATURA ---
def generate_text(model, start_string, temperature=0.7, num_generate=400):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []

    # Disattiviamo i gradienti per risparmiare memoria durante l'inferenza
    with torch.no_grad():
        for i in range(num_generate):
            predictions = model(input_eval)
            # Applichiamo la temperatura ai logits
            predictions = predictions[:, -1, :] / temperature

            predicted_id_tensor = keras.random.categorical(predictions, num_samples=1)
            predicted_id = int(keras.ops.convert_to_numpy(predicted_id_tensor)[0, 0])

            new_char_tensor = tf.expand_dims([predicted_id], 0)
            input_eval = tf.concat([input_eval, new_char_tensor], axis=-1)

            # Sliding window per mantenere il contesto
            if input_eval.shape[1] > seq_length:
                input_eval = input_eval[:, 1:]

            text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

In [12]:
# --- LOOP DI SPERIMENTAZIONE ---
temperatures = [0.1, 1.0, 2.0]
seed = "KING LEAR: "

print(f"\n{'='*50}")
print(f"ANALISI DELLA PERSONALITÀ DEL MODELLO (Seed: {seed})")
print(f"{'='*50}\n")

for temp in temperatures:
    print(f"--- TEST CON TEMPERATURA: {temp} ---")
    generated = generate_text(model, start_string=seed, temperature=temp, num_generate=400)
    print(generated)
    print(f"\n{'-'*50}\n")


ANALISI DELLA PERSONALITÀ DEL MODELLO (Seed: KING LEAR: )

--- TEST CON TEMPERATURA: 0.1 ---
KING LEAR: I say, I have some to thee.

GRUMIO:
A goodly sons, and so shall I say.

PETRUCHIO:
Why, then we will not be so fair and the world.

GRUMIO:
A goodly sir, that I was a son of my son,
I would not so make thee to the sea that she shall stay.

GRUMIO:
And then I say, I will not so so much of the world.

QUEEN ELIZABETH:
And therefore I should see the common with thee
That she shall strew the sea that

--------------------------------------------------

--- TEST CON TEMPERATURA: 1.0 ---
KING LEAR: but love advisaster bettor,
I cannot your grace than Patricians for you,
May nouristers in how, yellow of purse
From Padua. Py and the eid my kindmen of my tongures
Be luggives for the seaf that God's squarrel,
It island more than thou blust: uiship, beized you stand
Would see her body well, were I have his of compise;
For still I'll have scrute's gone.

CLAUDIO:
I am ransed itself safe so.
Th

**Caratteri unici Shakespeare:** 65

---
# Capitolo 2 — Dante Alighieri

## 2.1 Download del testo
`get_file` di Keras dà problemi con link esterni: usiamo `requests`. Attenzione all'encoding: `utf-8` per gli accenti (à, è, ì, ò, ù).

In [13]:
# Codice suggerito dalla consegna

# Disabilitiamo i messaggi di avviso "InsecureRequestWarning" per avere un output pulito
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def scarica_testo_dante():
    """
    Scarica la Divina Commedia ignorando gli errori di certificato SSL.
    """
    url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"

    print(f"Scaricamento testo da: {url}...")
    try:
        # AGGIUNTA FONDAMENTALE: verify=False
        # Dice a Python: "Fidati del server anche se non riconosci il certificato"
        response = requests.get(url, verify=False)
        response.raise_for_status()

        response.encoding = 'utf-8'
        text = response.text

        print(f"Download completato. Lunghezza testo: {len(text)} caratteri")
        print(f"Esempio (primi 100 caratteri):\n{text[:100]}")
        return text

    except Exception as e:
        print(f"Errore ancora presente: {e}")
        # Se fallisce anche così, usiamo un link alternativo di GitHub (spesso più compatibile)
        print("Tentativo con link di riserva (GitHub)...")
        try:
            url_backup = "https://raw.githubusercontent.com/wpm/t-sne-text-vis/master/data/divina_commedia.txt"
            r2 = requests.get(url_backup, verify=False)
            r2.encoding = 'utf-8'
            return r2.text
        except:
            return None

# Eseguiamo il download
text = scarica_testo_dante()

Scaricamento testo da: https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt...
Download completato. Lunghezza testo: 551846 caratteri
Esempio (primi 100 caratteri):
LA DIVINA COMMEDIA
di Dante Alighieri
INFERNO



Inferno: Canto I

  Nel mezzo del cammin di


## 2.2 Vocabolario e vettorizzazione
Dizionari carattere ↔ intero. Confronta il numero di caratteri unici con quello di Shakespeare.

In [17]:
vocab = sorted(set(text))
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])

print(f"Il testo contiene {len(vocab)} caratteri unici.")
print(vocab)
print(f"Testo vettorizzato: {text_as_int[:10]}")


Il testo contiene 69 caratteri unici.
['\n', '\r', ' ', '!', '"', "'", '(', ')', ',', '-', '.', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'X', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z', '~', 'à', 'è', 'é', 'ì', 'ï', 'ò', 'ó', 'ù']
Testo vettorizzato: [23 14  2 17 22 33 22 25 14  2]


---
# Capitolo 3 — Adattamento dei parametri: `seq_length`

Le terzine sono più brevi dei monologhi inglesi: scendi da 100 a **80** o **60**.

*Riflessione:* come cambia la velocità di ogni epoca?

In [19]:
seq_length = 60
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

BATCH_SIZE = 64
BUFFER_SIZE = 10_000
dataset = dataset.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
print(f"Dataset pronto. Batch shape: {dataset.element_spec}")


Dataset pronto. Batch shape: (TensorSpec(shape=(64, 60), dtype=tf.int64, name=None), TensorSpec(shape=(64, 60), dtype=tf.int64, name=None))


---
# Capitolo 4 — Modello e Training

## 4.1 Costruzione del modello
Architettura invariata rispetto a Shakespeare: Embedding + LSTM + Dense.

In [21]:

# --- DEFINIZIONE DEL MODELLO (EMBEDDING + 1 LSTM STATEFUL) ---
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

def costruisci_modello(vocab_size, embedding_dim, rnn_units, batch_size):
    model = keras.Sequential([
        keras.layers.Embedding(vocab_size, embedding_dim),

        keras.layers.LSTM(rnn_units,
                          return_sequences=True,
                          stateful=True,
                          recurrent_initializer='glorot_uniform'),

        keras.layers.Dense(vocab_size)
    ])

    # (Batch Size, Lunghezza Sequenza variabile)
    model.build(input_shape=(batch_size, None))

    return model

model = costruisci_modello(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)
model.summary()

# Test rapido sulla forma dell'output
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(f"Output shape (batch, seq, vocab): {example_batch_predictions.shape}")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (64, None, 256)        │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (64, None, 1024)       │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (64, None, 69)         │        70,725 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,335,365 (20.35 MB)

 Trainable params: 5,335,365 (20.35 MB)

 Non-trainable params: 0 (0.00 B)

Output shape (batch, seq, vocab): (64, 60, 69)


## 4.2 Addestramento
Monitora la `loss`: se non scende abbastanza aumenta `epochs` (almeno 10-20 per risultati leggibili).

In [22]:
# 1. Loss: sparse categorical crossentropy con from_logits=True
# 2. compile con optimizer 'adam'
# 3. Callback ModelCheckpoint (solo pesi, estensione .weights.h5) in ./training_checkpoints_dante
# 4. fit sul dataset con EPOCHS

# --- TRAINING ---
model.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

checkpoint_dir = './training_checkpoints_dante'

checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True,
    verbose=1)

EPOCHS_DANTE = 20
history = model.fit(dataset, epochs=EPOCHS_DANTE, callbacks=[checkpoint_callback])


Epoch 1/20


c:\Users\veron\miniconda3\envs\epicode-io\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 700ms/step - loss: 2.5721
Epoch 1: saving model to ./training_checkpoints_dante\ckpt_1.weights.h5

Epoch 1: finished saving model to ./training_checkpoints_dante\ckpt_1.weights.h5
141/141 ━━━━━━━━━━━━━━━━━━━━ 101s 702ms/step - loss: 2.5721
Epoch 2/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 733ms/step - loss: 2.0015
Epoch 2: saving model to ./training_checkpoints_dante\ckpt_2.weights.h5

Epoch 2: finished saving model to ./training_checkpoints_dante\ckpt_2.weights.h5
141/141 ━━━━━━━━━━━━━━━━━━━━ 103s 734ms/step - loss: 2.0015
Epoch 3/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 737ms/step - loss: 1.8326
Epoch 3: saving model to ./training_checkpoints_dante\ckpt_3.weights.h5

Epoch 3: finished saving model to ./training_checkpoints_dante\ckpt_3.weights.h5
141/141 ━━━━━━━━━━━━━━━━━━━━ 104s 738ms/step - loss: 1.8326
Epoch 4/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 734ms/step - loss: 1.7251
Epoch 4: saving model to ./training_checkpoints_dante\ckpt_4.weights.h5

Epoch 4: finished sa

---
# Capitolo 5 — Generazione e Tuning della Temperatura

## 5.1 Funzione di generazione

In [23]:
# Ricostruzione del modello per inferenza (Batch size = 1)
model_gen = costruisci_modello(vocab_size, embedding_dim, rnn_units, batch_size=1)

# Caricamento dei pesi dell'ultima epoca salvata
files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.weights.h5')]
latest_file = max([os.path.join(checkpoint_dir, f) for f in files], key=os.path.getctime)
print(f"Caricamento pesi da: {latest_file}")
model_gen.load_weights(latest_file)

def generate_text(model, start_string, temperature=0.7, num_generate=400):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []

def genera_testo(model, start_string, temperature=1.0, num_generate=500):
    # Convertiamo la stringa iniziale in numeri
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    for layer in model.layers:
        if hasattr(layer, 'reset_states'):
            layer.reset_states()

    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = predictions[:, -1, :] / temperature

        predicted_id_tensor = keras.random.categorical(predictions, num_samples=1)
        predicted_id = int(keras.ops.convert_to_numpy(predicted_id_tensor)[0, 0])

        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))


Caricamento pesi da: ./training_checkpoints_dante\ckpt_20.weights.h5


## 5.2 Esperimento: prompt dantesco + tre temperature
- **0.2 (freddo):** entra in loop ripetendo la stessa parola o frase?
- **1.0 (standard):** sintassi e grammatica reggono?
- **2.0 (folle):** parole inventate? Lingua aliena?

In [25]:
temperatures = [0.2, 1.0, 2.0]
seeds = ["Nel mezzo ", "Selva ", "  Virgilio "]

for seed in seeds:
    print(f"\n{'='*50}")
    print(f"ANALISI DELLA PERSONALITÀ DEL MODELLO (Seed: {seed})")
    print(f"{'='*50}\n")

    for temp in temperatures:
        print(f"--- TEST CON TEMPERATURA: {temp} ---")
        generated = genera_testo(model_gen, start_string=seed, temperature=temp, num_generate=400)
        print(generated)
        print(f"\n{'-'*50}\n")



ANALISI DELLA PERSONALITÀ DEL MODELLO (Seed: Nel mezzo )

--- TEST CON TEMPERATURA: 0.2 ---
Nel mezzo stelle
che sentiva solo in su la risposta,
come la vergine del mar de la spada
che la terra che s'apparve la via di sua vita
di quella gente che per lo scoglio
che solo a l'altro schermo l'anima condizione.
  Così disse 'l maestro mio disse: "Vieni a lei la testa
di quel che si stende, e la mia donna
di quel che tu vedi, e questo modo
che fa sì come s'addorna in su la riva
de la sua sper

--------------------------------------------------

--- TEST CON TEMPERATURA: 1.0 ---
Nel mezzo stesso è Amoro.
  Manidia Corsecca.
  Qualunque già queste cose dietro
un peccatore intelletto.
  Indi sovra 'l vero amai più c'aspetta,
con piangere infan da Luche sapecchiede,
non che parlar tutto loco
là giù per tortiti, fregò ciò ch'i' mi sentì ampini
Mantano,
che dal suo gravertà che m'intende.
  Dentro e qua la gente di che vivo di sé stesso
l'altre volte
sanzi la mascedebra che 

-------------------

# Conclusioni

## 1. Quale temperatura ha generato la "terzina" più credibile?

**Temperatura 0.2.** Su tutti e tre i prompt produce versi grammaticalmente corretti, con lessico e movenze dantesche ("Così disse 'l maestro mio", "la mia donna", "la via scorta") e una struttura a versi brevi con il rientro tipico delle terzine.

Il prezzo è la ripetitività: non entra in loop dentro lo stesso testo (come Shakespeare a 0.1), ma ricicla formule fisse da un prompt all'altro: "la sua spera e con la scala si spense" compare quasi identica con "Selva" e con "Virgilio"; "che la testa", "di quel che", "che si distende" ritornano ovunque.

- **1.0:** la struttura in versi regge, ma compaiono parole inventate plausibili ("Manidia", "vincerando", "aboccidar") e la sintassi si perde. Più creativo, meno credibile.
- **2.0:** lingua aliena. Parole spezzate, maiuscole e simboli casuali (`~`, `"`), righe vuote: la distribuzione è così appiattita che anche i caratteri rarissimi vengono scelti spesso.

Limite comune a tutte le temperature: il modello imita lessico e ritmo, ma non produce né rima incatenata (ABA BCB) né endecasillabi. A livello di carattere, con 20 epoche, la metrica non viene appresa.

## 2. Miglior testo generato

Prompt `"  Virgilio "`, temperatura 0.2:

```
  Virgilio fui del mondo in su la terra di sopra e 'l vero amaro.
  La bestia che tutto s'avvallar seguita
con la prima cornice che la faccia stella
che la sua spera e la scala si spense,
e disse: "Vien che sen va con la sua famiglia
che la madre che più non si spazia.
  Questi che l'anima sola tra 'l giusto suo sermone
per la vista che tutto il mondo sentire,
  per lo suo fattor del ponte antica
```

## Osservazioni

**Prompt engineering.** Con il prompt `"Virgilio "` il modello generava solo caratteri casuali (`((y(yZ(x...`) anche a temperatura 0.2. Ogni prompt che inizia con "V" partendo da stato azzerato porta il modello fuori distribuzione: la "V" maiuscola nel testo compare anche nei titoli dei canti in numeri romani, e con `stateful=True` il modello in training non parte quasi mai da stato zero. Aggiungendo due spazi iniziali (`"  Virgilio "`, come l'inizio delle terzine nel testo) l'output torna sensato. Lo stesso modello, con un prompt diverso solo per due spazi, passa da inutilizzabile a credibile.

**Shakespeare vs Dante**

| | Shakespeare | Dante |
|---|---|---|
| Caratteri unici | 65 | 69 (vocali accentate, `\r`) |
| seq_length / BATCH_SIZE | 250 / 128 | 60 / 64 |
| Modello | 3 GRU × 1024, non stateful | 1 LSTM × 1024, stateful |
| Step per epoca | 34 | 141 |
| Durata epoca | 299 s (9 s/step) | 101 s (716 ms/step) |
| Epoche / loss finale | 10 / 1.3008 | 20 / 1.2143 |

**seq_length e velocità.** Abbassare `seq_length` rende ogni step più leggero, ma aumenta il numero di step per epoca: il testo elaborato per epoca resta lo stesso. La riduzione del tempo per epoca rispetto a Shakespeare dipende soprattutto dal modello più leggero (una LSTM invece di tre GRU). Il guadagno più netto è in generazione: con `stateful=True` ogni passo elabora un solo carattere invece di rileggere tutta la finestra di contesto.
